In [12]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Code example: popcount

Given a 64-bit integer number, find the number of 1s in its binary representation.

Example 1:

Input: 59487

Output: 9

Explanation: 9487’s binary representation is 0b10110010100001111

In [13]:
render_code("popcounts/popcount.cpp", show=["#else //A", "#endif"])

// popcounts/popcount.cpp:87-97 (11 lines)
#else //A
inline int popcount(uint64_t x){
    int c=0;
    while(x) //while any bits are set not sure how you are getting the value
    {
       c += x & 1;
       x = x >> 1;
    }
    return c;
}
#endif

In [14]:
render_code("popcounts/popcount.cpp", show=["#ifdef B", "#else"])

// popcounts/popcount.cpp:32-48 (17 lines)
#ifdef B
inline int popcount(uint64_t x) {
    int c = 0;
    while(x) //while any bits are set not sure how you are getting the value
    {
        c += x & 1;
        x = x >> 1;
        c += x & 1;
        x = x >> 1;
        c += x & 1;
        x = x >> 1;
        c += x & 1;
        x = x >> 1;
    }
    return c;
}
#else

In [15]:
render_code("popcounts/popcount.cpp", show=["#ifdef C", "#else"])

// popcounts/popcount.cpp:20-31 (12 lines)
#ifdef C
inline int popcount(uint64_t x) {
     int c = 0;
     int table[16] = {0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4};
     while(x) {
            c += table[(x & 0xF)];
            x = x >> 4;
     }
     return c;
}

#else

In [16]:
render_code("popcounts/popcount.cpp", show=["#ifdef D", "#else"])

// popcounts/popcount.cpp:49-59 (11 lines)
#ifdef D
inline int popcount(uint64_t x) {
     int c = 0;
     int table[16] = {0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4};
     for (uint64_t i = 0; i < 16; i++) {
            c += table[(x & 0xF)];
            x = x >> 4;
     }
     return c;
}
#else

In [17]:
render_code("popcounts/popcount.cpp", show=["#ifdef E", "#else"])

// popcounts/popcount.cpp:60-87 (28 lines)
#ifdef E
inline int popcount(uint64_t x) {
     int c = 0;
     for (uint64_t i = 0; i < 16; i++) {
         switch((x & 0xF))
         {
             case 1: c+=1; break;
             case 2: c+=1; break;
             case 3: c+=2; break;
             case 4: c+=1; break;
             case 5: c+=2; break;
             case 6: c+=2; break;
             case 7: c+=3; break;
             case 8: c+=1; break;
             case 9: c+=2; break;
             case 10: c+=2; break;
             case 11: c+=3; break;
             case 12: c+=2; break;
             case 13: c+=3; break;
             case 14: c+=3; break;
             case 15: c+=4; break;
             default: break;
         }
         x = x >> 4;
     }
     return c;
}
#else //A

## Who has the best performance?

In [18]:
render_code("popcounts/popcount.cpp", show="main")

// popcounts/popcount.cpp:114-136 (23 lines)
int main(int argc, char *argv[]) {

     uint64_t key = 0xdeadbeef;
     char preamble[1024];
     char epilogue[1024];
     char header[1024];
     char stat_file[] = "stats.csv";
     int count = 1000000000;
     uint64_t sum = 0;
     perfstats_init();
     perfstats_enable(1);     
     for (int i=0; i < count; i++)
     { 
        sum += popcount (RandLFSR(key)); 
     }
     perfstats_disable(1);
     sprintf(epilogue,"\n");
     sprintf(preamble,"");
     perfstats_print(preamble, stat_file, epilogue);
     perfstats_deinit();
     printf("Result: %lu\n", sum);
     return sum;
}

In [ ]:
! make -C popcounts clean; make -C popcounts
! echo "Version,IC,Cycles,CPI,CT,ET,L1_dcache_miss_rate,L1_dcache_misses,L1_dcache_accesses,branc_miss_predict_rate,branch_misses,branches" > stats.csv
! echo "Version A"; echo -n "A," >> stats.csv; time ./popcounts/popcount_A
! echo "Version B"; echo -n "B," >> stats.csv; time ./popcounts/popcount_B
! echo "Version C"; echo -n "C," >> stats.csv; time ./popcounts/popcount_C
! echo "Version D"; echo -n "D," >> stats.csv; time ./popcounts/popcount_D
! echo "Version E"; echo -n "E," >> stats.csv; time ./popcounts/popcount_E

make: Entering directory '/nfshome/htseng/courses/CS203/demo/OoO_programming/popcounts'
rm -f popcount_A popcount_B popcount_C popcount_D popcount_SSE42 popcount_E
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/OoO_programming/popcounts'
make: Entering directory '/nfshome/htseng/courses/CS203/demo/OoO_programming/popcounts'
g++ -O3 -DHAVE_LINUX_PERF_EVENT_H -w popcount.cpp perfstats.o -o popcount_A
g++ -S -O3 -DHAVE_LINUX_PERF_EVENT_H -w popcount.cpp -o A.s
g++ -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DB popcount.cpp perfstats.o -o popcount_B
g++ -S -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DB popcount.cpp -o B.s
g++ -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DC popcount.cpp perfstats.o -o popcount_C
g++ -S -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DC popcount.cpp -o C.s
g++ -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DD popcount.cpp perfstats.o -o popcount_D
g++ -S -O3 -DHAVE_LINUX_PERF_EVENT_H -w -DD popcount.cpp -o D.s
g++ -O3 -DHAVE_LINUX_PERF_EVENT_H -w -m64 -msse4.2 -DSSE42 popcount.cpp perfstats.o -o popcou

In [ ]:
df = render_csv("stats.csv")
df = df.loc[df['Version'].isin(["A","B"])]
display_df_mono(df = df)

In [ ]:
compare([do_render_code("./popcounts/A.s",show=["L6","jne	.L6"]),do_render_code("./popcounts/B.s",show=["L6","jne	.L6"])])

In [ ]:
df = render_csv("stats.csv")
df = df.loc[df['Version'].isin(["A","B","C"])]
display_df_mono(df = df)

In [ ]:
compare([do_render_code("./popcounts/B.s",show=["L6","jne	.L6"]),do_render_code("./popcounts/C.s",show=["L6","jne	.L6"])])

In [ ]:
df = render_csv("stats.csv")
df = df.loc[df['Version'].isin(["A","B","C","D"])]
display_df_mono(df = df)

In [ ]:
compare([do_render_code("./popcounts/C.s",show=["L6","jne	.L6"]),do_render_code("./popcounts/D.s",show=["L5","jne	.L5"])])

In [ ]:
df = render_csv("stats.csv")
display_df_mono(df = df)

In [ ]:
compare([do_render_code("./popcounts/D.s",show=["L5","jne	.L5"]),do_render_code("./popcounts/E.s",show=["L12",".LFE1593:"])])

### Hardware acceleration!

In [ ]:
! echo -n "SSE," >> stats.csv; ./popcounts/popcount_SSE42
display_df_mono(render_csv("stats.csv"))